### The Orbital Model - Is used to ensure where the cluster is at any given point in time, since its placed at SSO - it orbits a given geographical location at a specific time in an earth day, so for tasks like constant monitoring, imaging, checking for changes it is ideal, the reason for placing it in SSO is because the solar activity is predictible, and a constant source of power for the GPU cluster, with not really exposed to eclipses and requiring a primary battery run technology to operate, offers cleaner and more reliable energy sources

In [1]:
from skyfield.api import load, EarthSatellite, wgs84
import datetime
import random

In [2]:
class OrbitalModel:
    def __init__(self):
        """
        Initializes the physics engine. 
        Skyfield needs to download planetary data (ephemeris) to know exactly where the Sun and Earth are
        """
        print("Initializing Orbital Physics Engine...")
        self.ts = load.timescale()
        # Load the planetary ephemeris (contains Sun/Earth positions for illumination math)
        self.eph = load('de421.bsp')

        # We need a Two-Line Element (TLE) set to define the orbit. 
        # Here we use a proxy TLE for a Sun-Synchronous polar satellite

        line1 = '1 23710U 95059A   23301.12345678  .00000123  00000-0  12345-4 0  9999'
        line2 = '2 23710  98.5780  80.1234 0001234  90.0000 270.0000 14.29812345123456'

        self.satellite = EarthSatellite(line1, line2, 'Tsukuyomi-1', self.ts)
        print("Tsukuyomi-1 Orbital parameters loaded.")
    def get_telemetry(self, sim_time: datetime.datetime) -> dict:
        """
        Calculates the exact state of the satellite at a specific moment in time.
        In your FastAPI loop, you will call this function every second.
        """

        # Convert Python datetime to Skyfield's Time format
        t = self.ts.from_datetime(sim_time)

        # Calculate the 3D position of the satellite relative to Earth's center
        geocentric = self.satellite.at(t)

        # --- 1. THE ILLUMINATION MATH ---
        # This is the most crucial part for our Power and Thermal models.
        # It calculates ray-tracing from the Sun to the Satellite, checking if Earth blocks it.
        is_sunlit = geocentric.is_sunlit(self.eph)

        # --- 2. THE POSITION MATH ---
        # We project the satellite straight down to the Earth's surface to get Lat/Lon/Altitude
        subpoint = wgs84.subpoint(geocentric)
        latitude = subpoint.latitude.degrees
        longitude = subpoint.longitude.degrees
        elevation_km = subpoint.elevation.km

        # --- 3. CHAOS INJECTION (DEBRIS) ---
        # A simple randomizer for the hackathon demo to simulate a 0.5% chance of a debris warning.
        # You will likely override this with a manual button on your frontend.
        debris_warning = random.random() < 0.005

        # --- 4. DATA PACKAGING ---
        return {
            "timestamp": sim_time.isoformat(),
            "orbit_type": "SSO (Dawn-Dusk)",
            "is_sunlit": bool(is_sunlit), # True = charging & heating, False = eclipse
            "latitude": round(latitude, 4),
            "longitude": round(longitude, 4),
            "altitude_km": round(elevation_km, 2),
            "hazard_debris_warning": debris_warning
        }

In [4]:
if __name__ == "__main__":
    import time
    
    # Initialize the engine
    orbital_engine = OrbitalModel()
    
    # Start the simulation at the current real-world time
    # (Must be timezone-aware for Skyfield, so we use UTC)
    current_sim_time = datetime.datetime.now(datetime.timezone.utc)
    
    print("\nStarting PHOENIX Orbital Spigot...")
    print("-" * 50)
    
    # Simulate 5 ticks (representing 5 real-time seconds of your FastAPI loop)
    # We fast-forward the simulation time by 10 minutes per tick just so you can see the numbers move.
    for i in range(5):
        # 1. Ask the physics engine for the current state
        telemetry = orbital_engine.get_telemetry(current_sim_time)
        
        # 2. Print it (In the real app, this is where you send the JSON to the frontend)
        print(f"Time: {telemetry['timestamp']}")
        print(f"Location: Lat {telemetry['latitude']}, Lon {telemetry['longitude']}, Alt {telemetry['altitude_km']} km")
        print(f"Sunlight Status: {'☀️ SUNLIT' if telemetry['is_sunlit'] else '🌑 ECLIPSE'}")
        print(f"Debris Warning: {'⚠️ YES' if telemetry['hazard_debris_warning'] else '✅ CLEAR'}")
        print("-" * 50)
        
        # 3. Advance the simulation clock by 10 minutes for the next loop
        current_sim_time += datetime.timedelta(minutes=1)
        
        # Real-world pause so your terminal doesn't explode
        time.sleep(1)

Initializing Orbital Physics Engine...
Tsukuyomi-1 Orbital parameters loaded.

Starting PHOENIX Orbital Spigot...
--------------------------------------------------
Time: 2026-06-10T09:38:59.983773+00:00
Location: Lat 8.9363, Lon 79.8964, Alt 793.49 km
Sunlight Status: ☀️ SUNLIT
Debris Warning: ✅ CLEAR
--------------------------------------------------
Time: 2026-06-10T09:39:59.983773+00:00
Location: Lat 5.3813, Lon 79.1043, Alt 793.68 km
Sunlight Status: ☀️ SUNLIT
Debris Warning: ✅ CLEAR
--------------------------------------------------
Time: 2026-06-10T09:40:59.983773+00:00
Location: Lat 1.8258, Lon 78.3185, Alt 794.02 km
Sunlight Status: ☀️ SUNLIT
Debris Warning: ✅ CLEAR
--------------------------------------------------
Time: 2026-06-10T09:41:59.983773+00:00
Location: Lat -1.7296, Lon 77.5349, Alt 794.49 km
Sunlight Status: ☀️ SUNLIT
Debris Warning: ✅ CLEAR
--------------------------------------------------
Time: 2026-06-10T09:42:59.983773+00:00
Location: Lat -5.2841, Lon 76.7493,